In [ ]:
# Synthetic Direct Postgres → Bronze Handoff

This notebook skips the Kafka staging and long-format melting path.

It takes the wide synthetic stream table already stored in Postgres and turns it into a single bronze-ready table that looks like the original pump dataset shape.

## What this notebook does
1. Reads one or more synthetic batches from Postgres.
2. Combines them into one stable ordered table.
3. Creates a unified row number and unified episode number.
4. Adds a fresh time index and timestamp series.
5. Maps synthetic state values into the original-style `machine_status` label.
6. Cuts the table down to the final bronze handoff structure.
7. Optionally writes the final result back to Postgres and/or parquet.

## Notes before you run this

- This notebook assumes your source table is already **wide** and contains the synthetic sensor columns plus metadata such as `batch_id`, `stream_state`, `phase`, and `meta__episode_id`.
- It is designed to handle **multiple appended batches** where episode numbering may restart inside each batch.
- The final dataframe can be kept in an **original-dataset style** (`timestamp`, `sensor_*`, `machine_status`) or keep extra lineage columns at the end for Bronze auditing.

In [ ]:
import logging
import json
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd

from utils.paths import get_paths
from utils.file_io import save_data
from utils.logging_setup import configure_logging, log_layer_paths
from utils.pipeline_config_loader import load_pipeline_config
from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
)
from utils.postgres_util import table_exists
from utils.synthetic_postgres_to_bronze import (
    build_engine_from_project_env,
    get_table_columns,
    get_sensor_columns,
    read_synthetic_stream_dataframe,
    validate_synthetic_stream_dataframe,
    build_bronze_handoff_from_postgres,
    summarize_bronze_handoff_dataframe,
    write_bronze_handoff_to_postgres,
)

In [ ]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# --- Direct handoff layer naming ---
DIRECT_LAYER_NAME = "synthetic_bronze_handoff"
PROCESS_RUN_ID = make_process_run_id("synthetic_to_bronze")

In [ ]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)
CONFIG = config_obj.data

PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})
PIPELINE_MODE = str(PIPELINE.get("execution_mode", "batch"))

DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = str(CONFIG["versions"]["truth"])

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])
LOGS_PATH = Path(PATHS["logs_root"])

TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

print("DATASET_NAME:", DATASET_NAME)
print("PIPELINE_MODE:", PIPELINE_MODE)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

In [ ]:
direct_log_path = paths.logs / "synthetic_postgres_to_bronze_no_kafka.log"

configure_logging(
    "capstone",
    direct_log_path,
    level=logging.INFO,
    overwrite_handlers=True,
)

logger = logging.getLogger("capstone.synthetic_to_bronze")
logger.info("Synthetic direct Postgres to Bronze handoff notebook starting")

log_layer_paths(paths, current_layer="synthetic", logger=logger)

In [ ]:
def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> Optional[str]:
    """
    Return the latest truth hash for a given layer + dataset from truth_index.jsonl.
    """
    if not truth_index_path.exists():
        return None

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            record_layer = str(record.get("layer_name", "")).strip().lower()
            record_dataset = str(record.get("dataset_name", "")).strip().lower()

            if record_layer == layer_name_norm and record_dataset == dataset_name_norm:
                latest_record = record

    if latest_record is None:
        return None

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        return None

    return str(truth_hash).strip()

In [ ]:
# ---------------------------
# Source Postgres settings
# ---------------------------
SOURCE_SCHEMA = "capstone"
SOURCE_TABLE = f"synthetic_{DATASET_NAME}_stream"

# Use None for all available batches, or a list like [1, 2, 3]
BATCH_IDS = None

# ---------------------------
# Final output shaping
# ---------------------------
START_TIMESTAMP = "2018-04-01 00:00:00"
SAMPLING_FREQUENCY = "1min"

TARGET_TOTAL_ROWS = None
TRIM_MODE = "head"   # head | tail | random

# Keep False for a strict original-like handoff
KEEP_LINEAGE_COLUMNS = True

# Optional synthetic helper flag
INCLUDE_SYNTHETIC_ANOMALY_FLAG = False

# Keep False unless you intentionally want every extra source column carried through
KEEP_OTHER_COLUMNS = False

# ---------------------------
# Postgres write-back
# ---------------------------
WRITE_TO_POSTGRES = True
TARGET_SCHEMA = "capstone"
TARGET_TABLE = f"synthetic_{DATASET_NAME}_bronze_handoff"
TARGET_IF_EXISTS = "replace"

# ---------------------------
# File export
# ---------------------------
EXPORT_PARQUET = True
EXPORT_CSV = False

PARQUET_OUTPUT_PATH = (
    ARTIFACTS_ROOT
    / "synthetic"
    / DATASET_NAME
    / f"{DATASET_NAME}__synthetic__bronze_handoff.parquet"
)

CSV_OUTPUT_PATH = (
    ARTIFACTS_ROOT
    / "synthetic"
    / DATASET_NAME
    / f"{DATASET_NAME}__synthetic__bronze_handoff.csv"
)

# ---------------------------
# Truth lineage
# ---------------------------
PARENT_TRUTH_HASH = get_latest_truth_hash(
    truth_index_path=TRUTH_INDEX_PATH,
    layer_name="synthetic",
    dataset_name=DATASET_NAME,
)

print("SOURCE:", f"{SOURCE_SCHEMA}.{SOURCE_TABLE}")
print("TARGET:", f"{TARGET_SCHEMA}.{TARGET_TABLE}")
print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)

## Connect to Postgres and inspect the source table

In [ ]:
engine = build_engine_from_project_env()

source_exists = table_exists(
    engine,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE,
)
print("Source table exists:", source_exists)

if not source_exists:
    raise ValueError(f"Source table not found: {SOURCE_SCHEMA}.{SOURCE_TABLE}")

source_columns = get_table_columns(
    engine,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE,
)

print("Source column count:", len(source_columns))
print("Sensor column count:", len(get_sensor_columns(source_columns)))
print("First 25 columns:", source_columns[:25])

logger.info(
    "Source table inspected | schema=%s | table=%s | column_count=%s | sensor_count=%s",
    SOURCE_SCHEMA,
    SOURCE_TABLE,
    len(source_columns),
    len(get_sensor_columns(source_columns)),
)

## Preview the raw source rows

This cell lets you confirm that the table already looks like the wide synthetic stream you expect before the bronze handoff logic runs.

In [ ]:
raw_stream_df = read_synthetic_stream_dataframe(
    engine,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE,
    batch_ids=BATCH_IDS,
)

validation_report = validate_synthetic_stream_dataframe(raw_stream_df)

print("Raw shape:", raw_stream_df.shape)
print("Validation report:")
print(json.dumps(validation_report, indent=2))

display(raw_stream_df.head())

logger.info(
    "Raw synthetic stream loaded | rows=%s | columns=%s",
    len(raw_stream_df),
    len(raw_stream_df.columns),
)

## Build the bronze-ready dataframe

This is the main conversion step.

In [ ]:
bronze_handoff_df = build_bronze_handoff_from_postgres(
    engine,
    source_schema=SOURCE_SCHEMA,
    source_table=SOURCE_TABLE,
    batch_ids=BATCH_IDS,
    start_timestamp=START_TIMESTAMP,
    frequency=SAMPLING_FREQUENCY,
    target_total_rows=TARGET_TOTAL_ROWS,
    trim_mode=TRIM_MODE,
    keep_lineage_columns=KEEP_LINEAGE_COLUMNS,
    include_anomaly_flag=INCLUDE_SYNTHETIC_ANOMALY_FLAG,
    keep_other_columns=KEEP_OTHER_COLUMNS,
)

bronze_summary = summarize_bronze_handoff_dataframe(bronze_handoff_df)

print("Bronze handoff summary:")
print(json.dumps(bronze_summary, indent=2))

logger.info("Bronze handoff summary: %s", bronze_summary)

## Review the final result
### Status distribution and batch coverage checks

In [ ]:
print("Final dataframe shape:", bronze_handoff_df.shape)
print("Final columns:")
print(bronze_handoff_df.columns.tolist())

display(bronze_handoff_df.head())

if "machine_status" in bronze_handoff_df.columns:
    print("\nmachine_status counts:")
    print(bronze_handoff_df["machine_status"].value_counts(dropna=False))

if "batch_id" in bronze_handoff_df.columns:
    print("\nrows by batch_id:")
    print(bronze_handoff_df["batch_id"].value_counts().sort_index())

if "meta__episode_id_unified" in bronze_handoff_df.columns:
    print("\nunified episode count:", bronze_handoff_df["meta__episode_id_unified"].nunique())

### Write to Postgres

In [ ]:
written_table_name = None

if WRITE_TO_POSTGRES:
    written_table_name = write_bronze_handoff_to_postgres(
        engine,
        bronze_handoff_df,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE,
        if_exists=TARGET_IF_EXISTS,
        logger=logger,
    )
    print(f"Wrote Postgres table: {TARGET_SCHEMA}.{written_table_name}")
else:
    print("WRITE_TO_POSTGRES is False")

In [ ]:
saved_parquet_path = None
saved_csv_path = None

if EXPORT_PARQUET:
    saved_parquet_path = save_data(
        bronze_handoff_df,
        PARQUET_OUTPUT_PATH,
        index=False,
    )
    print("Saved parquet:", saved_parquet_path)

if EXPORT_CSV:
    saved_csv_path = save_data(
        bronze_handoff_df,
        CSV_OUTPUT_PATH,
        index=False,
    )
    print("Saved csv:", saved_csv_path)

if not EXPORT_PARQUET and not EXPORT_CSV:
    print("No file export selected")

## Optional: export parquet / csv

In [ ]:
truth_base = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=DIRECT_LAYER_NAME,
    process_run_id=PROCESS_RUN_ID,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

truth_base = update_truth_section(
    truth_base,
    "config_snapshot",
    {
        "source_schema": SOURCE_SCHEMA,
        "source_table": SOURCE_TABLE,
        "batch_ids": BATCH_IDS,
        "start_timestamp": START_TIMESTAMP,
        "sampling_frequency": SAMPLING_FREQUENCY,
        "target_total_rows": TARGET_TOTAL_ROWS,
        "trim_mode": TRIM_MODE,
        "keep_lineage_columns": KEEP_LINEAGE_COLUMNS,
        "include_synthetic_anomaly_flag": INCLUDE_SYNTHETIC_ANOMALY_FLAG,
        "keep_other_columns": KEEP_OTHER_COLUMNS,
        "target_schema": TARGET_SCHEMA,
        "target_table": TARGET_TABLE,
        "write_to_postgres": WRITE_TO_POSTGRES,
        "target_if_exists": TARGET_IF_EXISTS,
        "export_parquet": EXPORT_PARQUET,
        "export_csv": EXPORT_CSV,
    },
)

truth_base = update_truth_section(
    truth_base,
    "runtime_facts",
    {
        "source_row_count": int(len(raw_stream_df)),
        "final_row_count": int(len(bronze_handoff_df)),
        "final_column_count": int(len(bronze_handoff_df.columns)),
        "sensor_column_count": int(len(get_sensor_columns(bronze_handoff_df.columns))),
        "machine_status_counts": bronze_summary.get("machine_status_counts", {}),
        "batch_count": bronze_summary.get("batch_count"),
        "episode_count_unified": bronze_summary.get("episode_count_unified"),
        "timestamp_min": bronze_summary.get("timestamp_min"),
        "timestamp_max": bronze_summary.get("timestamp_max"),
    },
)

truth_base = update_truth_section(
    truth_base,
    "artifact_paths",
    {
        "postgres_table": (
            f"{TARGET_SCHEMA}.{written_table_name}"
            if written_table_name is not None
            else None
        ),
        "parquet_output_path": str(saved_parquet_path) if saved_parquet_path is not None else None,
        "csv_output_path": str(saved_csv_path) if saved_csv_path is not None else None,
    },
)

meta_columns = [column for column in bronze_handoff_df.columns if str(column).startswith("meta__")]
meta_columns += [column for column in bronze_handoff_df.columns if column in {
    "unified_row_id",
    "observation_time_index",
    "batch_id",
    "row_in_batch",
    "global_cycle_id",
    "cycle_id",
    "global_row_id",
    "created_at",
    "stream_state",
    "phase",
}]
meta_columns = sorted(set(meta_columns))

feature_columns = [column for column in bronze_handoff_df.columns if column not in meta_columns]

truth_record = build_truth_record(
    truth_base=truth_base,
    row_count=len(bronze_handoff_df),
    column_count=len(bronze_handoff_df.columns),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=DIRECT_LAYER_NAME,
)

append_truth_index(
    truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

print("Truth saved to:", truth_path)
print("Truth hash:", truth_record["truth_hash"])

logger.info("Truth saved | path=%s | truth_hash=%s", truth_path, truth_record["truth_hash"])

## What to hand into Bronze next

At this point, `bronze_ready_df` should already be in the shape you want for a Bronze-style ingest handoff.

Typical next use:
- keep only original dataset columns if you want a very faithful raw handoff
- or keep the lineage columns for auditability while you test the new synthetic path